In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

**built-in aggregate func**

In [0]:
race_results_df = spark.read.parquet("abfss://presentation@databricksrg2026.dfs.core.windows.net/race_results")

In [0]:
display(race_results_df)

In [0]:
demo_df = race_results_df.filter("race_year = 2020")
display(demo_df)

In [0]:
demo_df_again = race_results_df.filter("race_year = 2020")
display(demo_df)

In [0]:
from pyspark.sql.functions import sum,avg,max,min,count,concat, countDistinct

In [0]:
demo_df.select(count("*")).show()

In [0]:
demo_df.select(countDistinct("race_name")).show()

In [0]:
demo_df.filter("driver_name = 'Lewis Hamilton'").select(sum("points"),countDistinct("race_name")).show()

**groupby**

In [0]:
group_demo = demo_df_again\
    .groupBy("driver_name")\
        .agg(countDistinct("race_name").alias("race_name") , sum("points").alias("no_of_races"))\
            .show()


In [0]:
display(demo_df)

**Windows func**

In [0]:
demo_df = race_results_df.filter("race_year in (2019, 2020)")

In [0]:
demo_group_df = demo_df\
    .groupBy("driver_name", "race_year")\
        .agg(countDistinct("race_name").alias("race_name") , sum("points").alias("no_of_races"))

In [0]:
display(demo_group_df)

In [0]:
from pyspark.sql.functions import window
from pyspark.sql.functions import desc

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
driverRankSpec = Window.partitionBy("race_year").orderBy(desc("no_of_races"))
final_df = demo_group_df.withColumn("rank",rank().over(driverRankSpec))
final_df.show()